# 01 — Faire mieux qu'une constante

**Question :** le contexte connu à l'avance (PDQ, mois, tendance) prédit-il mieux le nombre d'actes
de la semaine qu'une simple constante ?
On compare sur la **validation (2023)**, en changeant **une seule chose à la fois**.
Si un ajout n'aide pas, on garde le plus simple. Le **test reste fermé**.

**Contrat (v1, voir notebook 00) :** pour chaque PDQ actif, avant le début de la semaine,
estimer les actes déclarés de la semaine, à partir du calendrier seulement (PDQ, mois, temps écoulé).

In [1]:
import json
import hashlib
import tomllib
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
with open(ROOT / "configs/crimes.toml", "rb") as f:
    cfg = tomllib.load(f)

SEED = cfg["experiment"]["seed"]
DATA_PATH = ROOT / cfg["data"]["path"]
IN = ROOT / "artifacts" / "00_start_here"

panel = pd.read_parquet(IN / "panel.parquet")
meta = json.loads((IN / "meta.json").read_text())

# Provenance : le CSV a-t-il changé depuis la création du panel ?
empreinte = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()[:16]
print("Panel :", panel.shape, "| du", panel["SEMAINE"].min().date(), "au", panel["SEMAINE"].max().date())
print("Empreinte du CSV aujourd'hui :", empreinte, "| à la création du panel :", meta["source_sha16"])
assert empreinte == meta["source_sha16"], "Le CSV a changé depuis le notebook 00 : relance le 00 d'abord."

panel["ENSEMBLE"].value_counts()

Panel : (17835, 4) | du 2015-01-05 au 2026-10-12
Empreinte du CSV aujourd'hui : 8a28f8d8ad367780 | à la création du panel : 8a28f8d8ad367780


ENSEMBLE
train         12093
test           4234
validation     1508
Name: count, dtype: int64

## Variables connues à l'avance
`t` = années écoulées depuis la première semaine (la tendance) ; `mois` = 1 à 12 (la saison).
Les deux sont calculables avant le début de la semaine : elles respectent le contrat.

In [2]:
panel["t"] = (panel["SEMAINE"] - panel["SEMAINE"].min()).dt.days / 365.25
panel["mois"] = panel["SEMAINE"].dt.month

panel[["SEMAINE", "PDQ", "y", "t", "mois"]].head()

,SEMAINE,PDQ,y,t,mois
0,2015-01-05,1,1,0.0,1
1,2015-01-05,3,7,0.0,1
2,2015-01-05,4,13,0.0,1
3,2015-01-05,5,14,0.0,1
4,2015-01-05,7,21,0.0,1


## Outil d'évaluation
Une seule fonction entraîne et note **tous** les modèles de la même façon : la comparaison est équitable.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

train = panel[panel["ENSEMBLE"] == "train"]
validation = panel[panel["ENSEMBLE"] == "validation"]
# Pas de variable « test » : il reste fermé.

modeles = {}      # les modèles entraînés, par nom
resultats = []    # une ligne de notes par modèle

def evaluer(nom, estimateur, numeriques=(), categorielles=()):
    colonnes = list(numeriques) + list(categorielles)

    # 1. Préparation des colonnes
    etapes = []
    if numeriques:
        etapes.append(("num", StandardScaler(), list(numeriques)))
    if categorielles:
        etapes.append(("cat", OneHotEncoder(handle_unknown="ignore"), list(categorielles)))

    # 2. Préparation + modèle dans un seul pipeline, entraîné sur le TRAIN seulement
    modele = make_pipeline(ColumnTransformer(etapes), estimateur)
    modele.fit(train[colonnes], train["y"])

    # 3. Notes sur la validation (pour comparer) et sur le train (pour repérer le par-cœur)
    pred_validation = modele.predict(validation[colonnes])
    pred_train = modele.predict(train[colonnes])

    modeles[nom] = modele
    resultats.append({
        "modele": nom,
        "mae_validation": mean_absolute_error(validation["y"], pred_validation),
        "rmse_validation": root_mean_squared_error(validation["y"], pred_validation),
        "mae_train": mean_absolute_error(train["y"], pred_train),
    })
    return pd.DataFrame(resultats).set_index("modele").round(3)

## Référence : une constante
Avant tout vrai modèle : quelle erreur fait-on en prédisant la même valeur pour tous les PDQ et toutes les semaines ?

In [4]:
from sklearn.dummy import DummyRegressor

evaluer("moyenne", DummyRegressor(strategy="mean"), numeriques=["t"])
evaluer("mediane", DummyRegressor(strategy="median"), numeriques=["t"])

,mae_validation,rmse_validation,mae_train
modele,,,
moyenne,9.796,13.647,8.612
mediane,9.971,14.171,8.456
